# Notebook 3: Test Inference — 2.5D U-Net

Run inference on unseen test volumes from MSD Task03 Liver.

Outputs are saved to: `figures_v2/test_inference/{volume_name}/`

| Cell | What it does |
|---|---|
| 1 | Imports & device setup |
| 2 | Paths & config |
| 3 | Model definition & load |
| 4 | Select volume & run inference |
| 5 | Static visualisation — top-4 tumor-dense slices |
| 6 | HTML animation + summary table |
| 7 | All tumor-containing slices grid |
| 8 | Tumor burden profile |
| 9 | Orthogonal views (axial / coronal / sagittal) |
| 10 | Prediction confidence map |
| 11 | Per-slice class distribution chart |


## 1. Imports & Device Setup

In [ ]:
# ---- Standard library ----
import os

# ---- Numerical ----
import numpy as np

# ---- Plotting & animation ----
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
matplotlib.rcParams['animation.embed_limit'] = 100  # allow up to 100 MB

# ---- Jupyter display ----
from IPython.display import HTML, display

# ---- PyTorch ----
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- Device ----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device :', device)
if device.type == 'cuda':
    print('GPU          :', torch.cuda.get_device_name(0))


## 2. Paths & Config

Update these paths to match your local setup.

In [ ]:
TEST_IMG_CACHE = r"msd_liver_cache_test_v2\images"
BEST_PATH      = r"checkpoints_v2\unet_v2_best.pth"
FIG_DIR        = r"figures_v2"

os.makedirs(FIG_DIR, exist_ok=True)

NUM_CLASSES = 3   # 0=background, 1=liver, 2=tumor
IN_CHANNELS = 3   # 2.5D triplet

print('Paths ready.')
print(f'Base figure dir : {FIG_DIR}')


## 3. Model Definition & Load

Must match the architecture in Notebook 2 exactly.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c, dropout=0.0):
        super().__init__()
        layers = [
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.append(nn.Dropout2d(p=dropout))
        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_ch=3, n_cls=3, dropout=0.2):
        super().__init__()
        self.enc1       = DoubleConv(in_ch, 64)
        self.enc2       = DoubleConv(64,  128)
        self.enc3       = DoubleConv(128, 256)
        self.enc4       = DoubleConv(256, 512)
        self.pool       = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(512, 1024, dropout=dropout)
        self.up4        = nn.ConvTranspose2d(1024, 512, 2, 2)
        self.dec4       = DoubleConv(1024, 512)
        self.up3        = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec3       = DoubleConv(512,  256)
        self.up2        = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec2       = DoubleConv(256,  128)
        self.up1        = nn.ConvTranspose2d(128,  64, 2, 2)
        self.dec1       = DoubleConv(128,   64)
        self.out        = nn.Conv2d(64, n_cls, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b),  e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out(d1)


model = UNet(in_ch=IN_CHANNELS, n_cls=NUM_CLASSES).to(device)
model.load_state_dict(torch.load(BEST_PATH, map_location=device))
model.eval()
print(f'Loaded : {BEST_PATH}')
print(f'Params : {sum(p.numel() for p in model.parameters()):,}')


## 4. Select Volume & Run Inference

Change `VOLUME_IDX` to test a different patient volume.
All figures for this volume will be saved to `figures_v2/test_inference/{volume_name}/`.


In [ ]:
# ---- List available test volumes ----
test_files = sorted([f for f in os.listdir(TEST_IMG_CACHE) if f.endswith('.npy')])
print(f'Found {len(test_files)} test volumes:\n')
for i, f in enumerate(test_files):
    print(f'  [{i:>3}] {f}')




In [ ]:
# ---- Change this index to test a different volume ----
VOLUME_IDX = 50
# ---- Load selected volume ----
fname    = test_files[VOLUME_IDX]
vol_name = fname.replace('.npy', '')   # e.g. "liver_132"
volume   = np.load(os.path.join(TEST_IMG_CACHE, fname)).astype(np.float32)

# ---- Per-volume output directory ----
VOL_FIG_DIR = os.path.join(FIG_DIR, 'test_inference', vol_name)
os.makedirs(VOL_FIG_DIR, exist_ok=True)

print(f'Loaded     : {fname}')
print(f'Shape      : {volume.shape}  ->  {volume.shape[0]} slices x 3 channels x 256x256')
print(f'Output dir : {VOL_FIG_DIR}')

# ---- Run inference slice by slice, also collect softmax probs ----
preds = []
probs = []   # softmax probabilities per slice: (3, 256, 256)

with torch.no_grad():
    for z in range(volume.shape[0]):
        img    = torch.tensor(volume[z]).unsqueeze(0).to(device)   # (1,3,256,256)
        logits = model(img)                                         # (1,3,256,256)
        prob   = F.softmax(logits, dim=1).cpu().squeeze().numpy()   # (3,256,256)
        pred   = np.argmax(prob, axis=0)                            # (256,256)
        preds.append(pred)
        probs.append(prob)

preds = np.stack(preds)   # (N, 256, 256)
probs = np.stack(probs)   # (N, 3, 256, 256)

# ---- Compute summary stats ----
liver_slices = sum(1 for z in range(len(preds)) if (preds[z] == 1).sum() > 0)
tumor_slices = sum(1 for z in range(len(preds)) if (preds[z] == 2).sum() > 0)
tumor_counts = [(preds[z] == 2).sum() for z in range(len(preds))]
liver_counts = [(preds[z] == 1).sum() for z in range(len(preds))]
bg_counts    = [(preds[z] == 0).sum() for z in range(len(preds))]
top_slices   = sorted(range(len(tumor_counts)),
                      key=lambda z: tumor_counts[z], reverse=True)
tumor_slice_indices = [z for z in range(len(preds)) if (preds[z] == 2).sum() > 0]

print(f'\nInference complete.')
print(f'  Total slices     : {len(preds)}')
print(f'  Liver slices     : {liver_slices}')
print(f'  Tumor slices     : {tumor_slices}')
print(f'  Tumor detected   : {"Yes" if tumor_slices > 0 else "No"}')
print(f'  Most tumor-dense : slice {top_slices[0]}  ({tumor_counts[top_slices[0]]} px)')


## 5. Static Visualisation — Top-4 Tumor-Dense Slices

In [ ]:
n_show   = 4
show_idx = top_slices[:n_show]

fig, axes = plt.subplots(n_show, 3, figsize=(12, n_show * 4))
fig.suptitle(f'Top-{n_show} Tumor-Dense Slices — {fname}',
             fontsize=13, fontweight='bold')

col_titles = ['CT Slice', 'Predicted Mask', 'Overlay']
for ax, t in zip(axes[0], col_titles):
    ax.set_title(t, fontsize=11, fontweight='bold')

for row, z in enumerate(show_idx):
    ct_disp  = volume[z, 1]
    pred_z   = preds[z]
    tumor_px = (pred_z == 2).sum()
    liver_px = (pred_z == 1).sum()

    axes[row, 0].imshow(ct_disp, cmap='gray')
    axes[row, 0].set_ylabel(f'Slice {z}\nliver={liver_px}px\ntumor={tumor_px}px',
                             fontsize=8, labelpad=6)
    axes[row, 1].imshow(pred_z,  cmap='viridis', vmin=0, vmax=2)
    axes[row, 2].imshow(ct_disp, cmap='gray')
    axes[row, 2].imshow(pred_z,  cmap='jet', alpha=0.4, vmin=0, vmax=2)

    for ax in axes[row]:
        ax.axis('off')

plt.tight_layout()
out_path = os.path.join(VOL_FIG_DIR, 'top4_tumor_slices.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {out_path}')


## 6. HTML Animation + Summary Table

- `step=1` — every slice (slower to render)
- `step=2` — every other slice, faster (default)


In [ ]:
def run_animation(volume, preds, fname, step=2):

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    plt.subplots_adjust(top=0.85)

    im0 = axes[0].imshow(volume[0, 1], cmap='gray',    animated=True)
    im1 = axes[1].imshow(preds[0],     cmap='viridis', vmin=0, vmax=2, animated=True)
    im2 = axes[2].imshow(volume[0, 1], cmap='gray',    animated=True)
    ov  = axes[2].imshow(preds[0],     cmap='jet',     alpha=0.4, vmin=0, vmax=2, animated=True)

    axes[0].set_title('CT Slice',       fontsize=11, fontweight='bold')
    axes[1].set_title('Predicted Mask', fontsize=11, fontweight='bold')
    axes[2].set_title('Overlay',        fontsize=11, fontweight='bold')
    for ax in axes:
        ax.axis('off')

    title = fig.suptitle('', fontsize=12, fontweight='bold')

    def update(z):
        ct_disp  = volume[z, 1]
        pred_z   = preds[z]
        tumor_px = int((pred_z == 2).sum())
        liver_px = int((pred_z == 1).sum())
        im0.set_data(ct_disp)
        im1.set_data(pred_z)
        im2.set_data(ct_disp)
        ov.set_data(pred_z)
        tumor_label = '  [TUMOR]' if tumor_px > 0 else ''
        title.set_text(
            f'{fname}  -  Slice {z + 1}/{volume.shape[0]}{tumor_label}\n'
            f'liver={liver_px}px  |  tumor={tumor_px}px'
        )
        return im0, im1, im2, ov, title

    frames = range(0, volume.shape[0], step)
    anim   = FuncAnimation(fig, update, frames=frames, interval=120, blit=True)
    plt.close()

    # ---- Summary table ----
    total_tumor_px = int(sum(tumor_counts))
    print('=' * 47)
    print('  Inference Summary')
    print('=' * 47)
    print(f'  {"Volume":<28}: {fname}')
    print(f'  {"Output dir":<28}: {VOL_FIG_DIR}')
    print(f'  {"Total slices":<28}: {len(preds)}')
    print(f'  {"Liver slices":<28}: {liver_slices}')
    print(f'  {"Tumor slices":<28}: {tumor_slices}')
    print(f'  {"Tumor detected":<28}: {"Yes" if tumor_slices > 0 else "No"}')
    print(f'  {"Most tumor-dense slice":<28}: {top_slices[0]}  ({tumor_counts[top_slices[0]]} px)')
    print(f'  {"Total tumor pixels":<28}: {total_tumor_px:,}')
    print('=' * 47)

    return HTML(anim.to_jshtml())


run_animation(volume, preds, fname, step=2)


## 7. All Tumor-Containing Slices Grid

Shows every slice where the model predicted at least one tumor pixel.
Laid out in a 4-column grid of CT + overlay pairs.


In [ ]:
print(f'Tumor detected in {len(tumor_slice_indices)} slices: {tumor_slice_indices}\n')

if len(tumor_slice_indices) == 0:
    print('No tumor predicted in this volume.')
else:
    n_cols = 4
    n_rows = (len(tumor_slice_indices) + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols * 2, figsize=(n_cols * 2 * 3, n_rows * 3))
    if n_rows == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle(f'All Tumor-Containing Slices — {fname}',
                 fontsize=13, fontweight='bold')

    for col in range(n_cols):
        axes[0, col * 2    ].set_title(f'CT #{col+1}',      fontsize=9, fontweight='bold')
        axes[0, col * 2 + 1].set_title(f'Overlay #{col+1}', fontsize=9, fontweight='bold')

    for idx, z in enumerate(tumor_slice_indices):
        row      = idx // n_cols
        col_base = (idx %  n_cols) * 2
        ct_disp  = volume[z, 1]
        pred_z   = preds[z]
        tumor_px = int((pred_z == 2).sum())

        axes[row, col_base].imshow(ct_disp, cmap='gray')
        axes[row, col_base].set_ylabel(f'Slice {z}\ntumor={tumor_px}px',
                                        fontsize=7, labelpad=4)
        axes[row, col_base + 1].imshow(ct_disp, cmap='gray')
        axes[row, col_base + 1].imshow(pred_z,  cmap='jet', alpha=0.4, vmin=0, vmax=2)

        axes[row, col_base    ].axis('off')
        axes[row, col_base + 1].axis('off')

    for idx in range(len(tumor_slice_indices), n_rows * n_cols):
        row      = idx // n_cols
        col_base = (idx %  n_cols) * 2
        axes[row, col_base    ].axis('off')
        axes[row, col_base + 1].axis('off')

    plt.tight_layout()
    out_path = os.path.join(VOL_FIG_DIR, 'all_tumor_slices.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved -> {out_path}')


## 8. Tumor Burden Profile

Line chart of tumor pixel count per slice across the full volume.
Shows where in the volume tumor is concentrated, how many clusters exist,
and whether the lesion is focal or diffuse.


In [ ]:
slice_indices = np.arange(len(preds))
tumor_px_arr  = np.array(tumor_counts)
liver_px_arr  = np.array(liver_counts)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
fig.suptitle(f'Tumor Burden Profile — {fname}', fontsize=13, fontweight='bold')

# ---- Top: tumor pixel count per slice ----
axes[0].fill_between(slice_indices, tumor_px_arr, alpha=0.35, color='crimson')
axes[0].plot(slice_indices, tumor_px_arr, color='crimson', linewidth=1.5, label='Tumor pixels')
axes[0].set_ylabel('Tumor pixels / slice', fontsize=10)
axes[0].set_title('Predicted Tumor Pixel Count per Slice', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

peak_z  = top_slices[0]
peak_px = tumor_counts[peak_z]
axes[0].axvline(peak_z, color='darkred', linestyle='--', linewidth=1.2, alpha=0.7)
axes[0].annotate(f'Peak\nSlice {peak_z}\n({peak_px}px)',
                  xy=(peak_z, peak_px),
                  xytext=(peak_z + max(3, len(preds)//20), peak_px * 0.85),
                  fontsize=8, color='darkred',
                  arrowprops=dict(arrowstyle='->', color='darkred', lw=1.0))

# ---- Bottom: liver pixel count per slice (context) ----
axes[1].fill_between(slice_indices, liver_px_arr, alpha=0.25, color='steelblue')
axes[1].plot(slice_indices, liver_px_arr, color='steelblue', linewidth=1.5, label='Liver pixels')
axes[1].set_ylabel('Liver pixels / slice', fontsize=10)
axes[1].set_xlabel('Slice index (axial)', fontsize=10)
axes[1].set_title('Predicted Liver Pixel Count per Slice (context)', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

for z in tumor_slice_indices:
    axes[1].axvspan(z - 0.5, z + 0.5, color='crimson', alpha=0.25)

legend_patch = mpatches.Patch(color='crimson', alpha=0.25, label='Tumor slice')
axes[1].legend(handles=[axes[1].get_lines()[0], legend_patch], fontsize=9)

plt.tight_layout()
out_path = os.path.join(VOL_FIG_DIR, 'burden_profile.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {out_path}')


## 9. Orthogonal Views — Axial / Coronal / Sagittal

Shows 3 orthogonal cross-sections through the predicted segmentation volume.
All three views intersect at the most tumor-dense slice.

- **Axial** — standard top-down slice view
- **Coronal** — front-to-back cross section
- **Sagittal** — left-to-right cross section


In [ ]:
ct_3d   = volume[:, 1, :, :]   # (N, H, W) — centre channel
pred_3d = preds                 # (N, H, W)

z_mid = top_slices[0]
y_mid = ct_3d.shape[1] // 2
x_mid = ct_3d.shape[2] // 2

axial_ct      = ct_3d[z_mid,  :, :]
axial_pred    = pred_3d[z_mid, :, :]
coronal_ct    = ct_3d[:,  y_mid, :]
coronal_pred  = pred_3d[:, y_mid, :]
sagittal_ct   = ct_3d[:,  :, x_mid]
sagittal_pred = pred_3d[:, :, x_mid]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle(
    f'Orthogonal Views — {fname}\n'
    f'Intersection at axial={z_mid}, coronal y={y_mid}, sagittal x={x_mid}',
    fontsize=12, fontweight='bold'
)

view_data = [
    ('Axial',    f'Slice {z_mid}', axial_ct,    axial_pred),
    ('Coronal',  f'y = {y_mid}',   coronal_ct,  coronal_pred),
    ('Sagittal', f'x = {x_mid}',   sagittal_ct, sagittal_pred),
]

for col, (view_name, subtitle, ct_view, pred_view) in enumerate(view_data):
    axes[0, col].imshow(ct_view, cmap='gray', aspect='auto')
    axes[0, col].set_title(f'{view_name} — CT\n{subtitle}', fontsize=10, fontweight='bold')
    axes[0, col].axis('off')

    axes[1, col].imshow(ct_view,   cmap='gray', aspect='auto')
    axes[1, col].imshow(pred_view, cmap='jet',  alpha=0.4, vmin=0, vmax=2, aspect='auto')
    axes[1, col].set_title(f'{view_name} — Overlay', fontsize=10, fontweight='bold')
    axes[1, col].axis('off')

# Crosshair lines on axial view showing coronal/sagittal cut positions
for row in range(2):
    axes[row, 0].axhline(y_mid, color='yellow', linewidth=0.8, alpha=0.7)
    axes[row, 0].axvline(x_mid, color='yellow', linewidth=0.8, alpha=0.7)

plt.tight_layout()
out_path = os.path.join(VOL_FIG_DIR, 'orthogonal_views.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {out_path}')


## 10. Prediction Confidence Map

Shows softmax probability for each class as a heatmap.
Bright = high confidence, dark = uncertain boundary.
Shown for the most tumor-dense slice.


In [ ]:
z       = top_slices[0]
ct_disp = volume[z, 1]
prob_z  = probs[z]   # (3, 256, 256)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle(
    f'Prediction Confidence Map — {fname}  |  Slice {z}',
    fontsize=12, fontweight='bold'
)

class_names = ['Background', 'Liver', 'Tumor']
class_cmaps = ['Purples',    'Blues', 'Reds']

# Top row: CT, hard prediction, max confidence
axes[0, 0].imshow(ct_disp, cmap='gray')
axes[0, 0].set_title('CT Slice', fontsize=11, fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(preds[z], cmap='viridis', vmin=0, vmax=2)
axes[0, 1].set_title('Hard Prediction (argmax)', fontsize=11, fontweight='bold')
axes[0, 1].axis('off')

max_conf = prob_z.max(axis=0)   # (256, 256)
im = axes[0, 2].imshow(max_conf, cmap='hot', vmin=0.33, vmax=1.0)
axes[0, 2].set_title('Max Confidence per Pixel', fontsize=11, fontweight='bold')
axes[0, 2].axis('off')
plt.colorbar(im, ax=axes[0, 2], fraction=0.046, pad=0.04)

# Bottom row: per-class probability maps
for cls in range(3):
    im = axes[1, cls].imshow(prob_z[cls], cmap=class_cmaps[cls], vmin=0, vmax=1)
    axes[1, cls].set_title(f'{class_names[cls]} Probability', fontsize=11, fontweight='bold')
    axes[1, cls].axis('off')
    plt.colorbar(im, ax=axes[1, cls], fraction=0.046, pad=0.04)

plt.tight_layout()
out_path = os.path.join(VOL_FIG_DIR, 'confidence_map.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {out_path}')


## 11. Per-Slice Class Distribution

Stacked bar chart showing background / liver / tumor pixel counts for every slice.
Shows the full segmentation story across the volume at a glance.


In [ ]:
bg_arr    = np.array(bg_counts)
liver_arr = np.array(liver_counts)
tumor_arr = np.array(tumor_counts)
x         = np.arange(len(preds))

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle(f'Per-Slice Class Distribution — {fname}', fontsize=13, fontweight='bold')

# ---- Top: stacked bar chart ----
axes[0].bar(x, bg_arr,    label='Background', color='#4a4a6a', alpha=0.85)
axes[0].bar(x, liver_arr, label='Liver',      color='#2a9d8f', alpha=0.85, bottom=bg_arr)
axes[0].bar(x, tumor_arr, label='Tumor',      color='#e9c46a', alpha=0.95,
             bottom=bg_arr + liver_arr)
axes[0].set_ylabel('Pixel count', fontsize=10)
axes[0].set_title('Stacked Pixel Counts per Slice', fontsize=11)
axes[0].legend(fontsize=9, loc='upper right')
axes[0].grid(axis='y', alpha=0.3)

# ---- Bottom: tumor as % of slice ----
total_px  = bg_arr + liver_arr + tumor_arr
tumor_pct = np.where(total_px > 0, tumor_arr / total_px * 100, 0)

axes[1].fill_between(x, tumor_pct, alpha=0.4, color='crimson')
axes[1].plot(x, tumor_pct, color='crimson', linewidth=1.2)
axes[1].set_ylabel('Tumor % of slice', fontsize=10)
axes[1].set_xlabel('Slice index (axial)', fontsize=10)
axes[1].set_title('Tumor as % of Total Slice Pixels', fontsize=11)
axes[1].grid(True, alpha=0.3)

for z in tumor_slice_indices:
    axes[1].axvline(z, color='darkred', linewidth=0.6, alpha=0.5)

plt.tight_layout()
out_path = os.path.join(VOL_FIG_DIR, 'class_distribution.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {out_path}')
